# 00 · Descarga y consolidación de datos

Construye los dos datasets base del proyecto:

- **Retornos diarios reales** (hasta ~36 años, Norgate) para el universo
  reducido (25 bancos, backbone del predictor final) y el universo amplio
  (150 bancos, usado solo para entrenar los generadores con más datos).
- **Barras de 5 minutos reales** (EODHD, últimos ~2 años) y sus features
  intradía derivadas (volatilidad realizada, retorno de apertura/cierre,
  rango), para el universo amplio.

Todo se cachea en `datos/raw/` y `datos/interim/` (gitignored, pesan
demasiado y contienen la ruta al `APkey`); este notebook solo hay que
relanzarlo si cambia el universo, las fechas, o se borra la caché.

**Por qué dos universos distintos** (`src/config.py`): el predictor final
necesita ~30 años de retorno diario REAL por banco (solo 25 bancos los
tienen completos en el dump de Norgate). Los generadores, en cambio, solo
necesitan datos de los últimos 2 años — así que para darles más muestras
con las que aprender bien la distribución conjunta (retorno, features
intradía) se usan hasta 150 bancos, aunque no coticen desde 1990.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config, data_eodhd as de, data_norgate as dn, features as feat

## 1. Retornos diarios reales (Norgate)

Universo reducido (25 bancos, ~36 años completos).

In [2]:
prices_predictor = dn.load_daily_prices(config.PREDICTOR_TICKERS)
returns_predictor = dn.compute_log_returns(prices_predictor)
print("precios:", prices_predictor.shape, " retornos:", returns_predictor.shape)

cobertura = dn.coverage_report(prices_predictor)
cobertura

precios: (9169, 25)  retornos: (7865, 25)


,first_date,last_date,n_obs,pct_nan
ticker,,,,
BAC,1990-01-02,2026-05-29,9169,0.000000
WFC,1990-01-02,2026-05-29,9169,0.000000
JPM,1990-01-02,2026-05-29,9169,0.000000
C,1990-01-02,2026-05-29,9169,0.000000
HBAN,1990-01-02,2026-05-29,9169,0.000000
USB,1990-01-02,2026-05-29,9169,0.000000
TFC,1990-01-02,2026-05-29,9168,0.000109
KEY,1990-01-02,2026-05-29,9169,0.000000
RF,1990-01-02,2026-05-29,9169,0.000000


In [3]:
assert cobertura["pct_nan"].max() < 0.05, "Algún ticker del universo predictor tiene demasiados huecos"
cobertura.to_csv(config.TABLES_DIR / "00_cobertura_predictor.csv")

Universo amplio (150 bancos), retorno diario real solo desde el inicio de
la ventana real de 2 años (es lo único que necesitan los generadores).

In [4]:
prices_generator = dn.load_daily_prices(
    config.GENERATOR_TICKERS, start=config.REAL_INTRADAY_START_DATE
)
# dropna=None: NO exigimos que los 150 bancos coticen el mismo dia (varios
# no tienen historia completa a proposito, ver seccion introductoria). Cada
# ticker conserva sus propios NaN; build_conditional_pool los filtra 1 a 1.
returns_generator = dn.compute_log_returns(prices_generator, dropna=None)
print("precios:", prices_generator.shape, " retornos:", returns_generator.shape)
print(f"NaNs: {returns_generator.isna().mean().mean():.1%} de media por ticker (normal: altas/bajas parciales)")

precios: (502, 150)  retornos: (502, 150)
NaNs: 2.1% de media por ticker (normal: altas/bajas parciales)


## 2. Barras de 5 minutos reales (EODHD)

Se descargan (o se leen de caché) para el universo amplio. La API key se
lee de `datos/APkey` (gitignored) y nunca se imprime.

In [5]:
bars_by_ticker = de.download_universe_5m(tickers=config.GENERATOR_TICKERS)

BAC     114925 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WFC     114927 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
JPM     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
C       114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]


HBAN    114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
USB     114922 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TFC     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
KEY     114925 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
RF      114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FITB    114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
VLY     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FHN     114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CFG     114917 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FLG      36145 barras  [2024-10-28 13:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PNC     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
ONB     114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FNB     114927 barras  [2020-11-02 14:30

ASB     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WBS     114921 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 19:55:00+00:00]
BANC    114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
MTB     114922 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
ZION    114922 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]


NWBI    114915 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
SFNC    114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FFIN    114898 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
AUB     114927 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
UBSI    114924 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
EWBC    114925 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CVBF    114915 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CBSH    114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CFFN    114912 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PFS     114916 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
GBCI    114924 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
HOPE    114914 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FHB     114927 barras  [2020-11-02 14:30

WSBC    114903 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
RNST    114913 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PB      114921 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BRBS    112163 barras  [2020-12-16 14:45:00+00:00 -> 2026-08-28 20:00:00+00:00]
BUSE    114870 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BBT      19635 barras  [2025-09-02 13:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
HWC     114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TOWN    114874 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PNFP    114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WAFD    114906 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
UMBF    114875 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BKU     114925 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
ABCB    114880 barras  [2020-11-02 14:30

WSFS    114826 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
SFBS    114833 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FBK     114886 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
OSBC    114893 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CBU     114791 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NBTB    114785 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
HBNC    114897 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
STEL     75252 barras  [2022-10-03 13:30:00+00:00 -> 2026-07-30 19:55:00+00:00]
CNOB    114756 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
INDB    114848 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
LOB     114862 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BY      114779 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NBBK     52457 barras  [2023-12-28 14:30

PEBO    114805 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NPB      30125 barras  [2025-02-13 21:00:00+00:00 -> 2026-08-28 20:00:00+00:00]
CUBI    114846 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BANR    114746 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
HFWA    114660 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FFIC    111448 barras  [2020-11-02 14:30:00+00:00 -> 2026-06-30 19:55:00+00:00]
SHBI    114711 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BANF    114575 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TCBK    114499 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BCAL    111650 barras  [2020-12-16 15:45:00+00:00 -> 2026-08-28 20:00:00+00:00]
SICP       392 barras  [2026-06-18 20:00:00+00:00 -> 2026-08-28 19:55:00+00:00]
HBT     114338 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
OBK      64365 barras  [2023-05-22 13:30

BWB     114695 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CPF     114681 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
LKFN    114565 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PBFS    114208 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WABC    114621 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FRBA    114357 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FRST    114536 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
SRCE    114535 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FMBH    114478 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PDLB    114402 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TFIN     73624 barras  [2022-12-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
MPB     114617 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CASH    114761 barras  [2020-11-02 14:30

In [6]:
n_ok = sum(1 for df in bars_by_ticker.values() if len(df) > 0)
print(f"{n_ok}/{len(bars_by_ticker)} tickers con barras de 5 min descargadas")

cobertura_intradia = pd.DataFrame(
    {
        "ticker": list(bars_by_ticker.keys()),
        "n_bars": [len(df) for df in bars_by_ticker.values()],
        "first": [df.index.min() if len(df) else pd.NaT for df in bars_by_ticker.values()],
        "last": [df.index.max() if len(df) else pd.NaT for df in bars_by_ticker.values()],
    }
).set_index("ticker")
cobertura_intradia.to_csv(config.TABLES_DIR / "00_cobertura_intradia.csv")
cobertura_intradia.sort_values("n_bars").head(10)

150/150 tickers con barras de 5 min descargadas


,n_bars,first,last
ticker,,,
SICP,392,2026-06-18 20:00:00+00:00,2026-08-28 19:55:00+00:00
BBT,19635,2025-09-02 13:30:00+00:00,2026-08-28 20:00:00+00:00
NPB,30125,2025-02-13 21:00:00+00:00,2026-08-28 20:00:00+00:00
FBLA,35887,2024-10-23 14:45:00+00:00,2026-08-28 20:00:00+00:00
FLG,36145,2024-10-28 13:30:00+00:00,2026-08-28 20:00:00+00:00
UCB,40719,2024-08-06 13:30:00+00:00,2026-08-28 20:00:00+00:00
NBBK,52457,2023-12-28 14:30:00+00:00,2026-08-28 20:00:00+00:00
OBK,64365,2023-05-22 13:30:00+00:00,2026-08-28 20:00:00+00:00
MCHB,71590,2021-01-14 16:40:00+00:00,2026-08-28 20:00:00+00:00


## 3. Features intradía diarias (volatilidad realizada, etc.)

Primero se descarta, DÍA A DÍA, cualquier sesión con menos de
`MIN_BARS_PER_SESSION` barras (feed caído, apertura tardía — no cierres
anticipados legítimos por festivo, esos sí se quedan). Después, un ticker
entra en el pool de entrenamiento de los generadores solo si le quedan al
menos `MIN_SESSIONS_FOR_GENERATOR_POOL` sesiones válidas (filtra bancos
intervenidos/fusionados a mitad de la ventana real).

In [7]:
intraday_feats_all = {
    tk: feat.daily_intraday_features(bars) for tk, bars in bars_by_ticker.items()
}
intraday_feats_all = {
    tk: f[f["n_bars"] >= config.MIN_BARS_PER_SESSION] for tk, f in intraday_feats_all.items()
}
intraday_feats = {
    tk: f for tk, f in intraday_feats_all.items()
    if len(f) >= config.MIN_SESSIONS_FOR_GENERATOR_POOL
}
print(
    f"{len(intraday_feats)}/{len(intraday_feats_all)} tickers con "
    f">= {config.MIN_SESSIONS_FOR_GENERATOR_POOL} sesiones intradía válidas"
)

149/150 tickers con >= 60 sesiones intradía válidas


## 4. Pool condicional (retorno diario real, features intradía reales)

Dataset de entrenamiento de los 4 generadores del notebook 02: cada fila
es un día real de un banco cualquiera del universo amplio, con su retorno
diario y sus 4 features intradía reales. Cuantas más muestras, mejor
generalizan los generadores — de ahí usar el universo amplio.

In [8]:
pool, pool_meta = feat.build_conditional_pool(returns_generator, intraday_feats)
print("pool:", pool.shape, "  columnas:", ["log_return", *feat.INTRADAY_FEATURE_COLS])
pool_meta["ticker"].value_counts().describe()

pool: (53282, 5)   columnas: ['log_return', 'realized_vol', 'open_30m_ret', 'close_30m_ret', 'hl_range']


.../ipykernel/236143455.py:1: UserWarning: build_conditional_pool: descartadas 5/53287 filas (0.01%) por valores fuera de rango plausible (tickers probablemente en proceso de quiebra/exclusion con cruces erraticos, no volatilidad real de mercado). Ver config.POOL_MAX_*.
  pool, pool_meta = feat.build_conditional_pool(returns_generator, intraday_feats)


count    148.000000
mean     360.013514
std      141.309377
min        1.000000
25%      258.250000
50%      402.000000
75%      489.000000
max      501.000000
Name: count, dtype: float64

In [9]:
assert len(pool) > 5000, "Pool de entrenamiento de los generadores demasiado pequeño"

## 5. Guardar datasets intermedios (`datos/interim/`, gitignored)

In [10]:
np.save(config.INTERIM_DIR / "conditional_pool.npy", pool)
pool_meta.to_parquet(config.INTERIM_DIR / "conditional_pool_meta.parquet")
returns_predictor.to_parquet(config.INTERIM_DIR / "returns_predictor.parquet")
returns_generator.to_parquet(config.INTERIM_DIR / "returns_generator.parquet")

intraday_long = (
    pd.concat([f.assign(ticker=tk) for tk, f in intraday_feats.items()])
    .rename_axis("date")
    .reset_index()
)
intraday_long.to_parquet(config.INTERIM_DIR / "intraday_features_real.parquet")

print("Guardado en", config.INTERIM_DIR)

Guardado en datos/interim
